# Interview Environment vLLM Demo

This notebook runs the `Interview` environment with actual `vllm` agents.

It does four things:
- loads a local/chat model through `vllm`
- uses that model to generate the pre-offer back-and-forth during `env.reset()`
- shows the exact final candidate prompt after the generated dialogue
- lets the candidate model send the final response and resolves the episode

Use a vLLM-capable kernel for this notebook, such as `Python (deception-vllm)`.
        

In [ ]:
import os
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD', 'spawn')
# os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import importlib
import json
import re
import sys
from pathlib import Path
from pprint import pprint

import torch
from vllm import LLM, SamplingParams

ROOT = Path('/playpen-ssd/smerrill/deception2')
INTERVIEW_ROOT = ROOT / 'Interview'
INTERVIEW_SRC = INTERVIEW_ROOT / 'src'

if str(INTERVIEW_SRC) not in sys.path:
    sys.path.insert(0, str(INTERVIEW_SRC))

import interview_environment as interview_env
importlib.reload(interview_env)

JobInterviewEnvironment = interview_env.JobInterviewEnvironment
InterviewSpec = interview_env.InterviewSpec
        

In [ ]:
MODEL_NAME = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
SEED = 0
SCENARIO_NAME = 'ml_platform_engineer'
PRIVATE_PROFILE_NAME = 'no_offer'
GENERATED_DIALOGUE_TURNS = 4
HISTORY_WINDOW = 10

DTYPE = 'bfloat16'
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = 0.9
TRUST_REMOTE_CODE = False
TENSOR_PARALLEL_SIZE = max(1, torch.cuda.device_count())

DIALOGUE_TEMPERATURE = 0.8
FINAL_TEMPERATURE = 0.2
TOP_P = 0.95
DIALOGUE_MAX_TOKENS = 220
FINAL_MAX_TOKENS = 320

print('cuda_device_count =', torch.cuda.device_count())
print('tensor_parallel_size =', TENSOR_PARALLEL_SIZE)
print('model_name =', MODEL_NAME)
        

In [ ]:
def maybe_parse_json_object(value):
    if isinstance(value, dict):
        return value
    if value is None:
        return None

    text = str(value).strip()
    if not text:
        return None

    candidates = [text]
    match = re.search(r'\{.*\}', text, flags=re.S)
    if match:
        candidates.append(match.group(0))

    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
        except Exception:
            continue
        if isinstance(parsed, dict):
            return parsed
    return None


def show_messages(messages):
    for idx, msg in enumerate(messages):
        print(f'===== MESSAGE {idx} / role={msg["role"]} =====')
        print(msg['content'])
        print()


def show_dialogue(dialogue_history):
    for item in dialogue_history:
        print(f"{item['speaker']}: {item['message']}")
        print()


class VLLMInterviewAgent:
    def __init__(
        self,
        name,
        model,
        model_name,
        base_seed=0,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        dialogue_temperature=0.8,
        final_temperature=0.2,
        top_p=0.95,
        dialogue_max_tokens=220,
        final_max_tokens=320,
    ):
        self.name = name
        self.model = model
        self.model_name = model_name
        self.base_seed = int(base_seed)
        self.reasoning_instruction = reasoning_instruction
        self.instruction_format = instruction_format
        self.dialogue_temperature = float(dialogue_temperature)
        self.final_temperature = float(final_temperature)
        self.top_p = float(top_p)
        self.dialogue_max_tokens = int(dialogue_max_tokens)
        self.final_max_tokens = int(final_max_tokens)
        self.call_log = []

    def _infer_stage(self, messages):
        merged = ' '.join(str(msg.get('content', '')) for msg in messages)
        if 'DIALOGUE_MESSAGE' in merged or 'pre-offer' in merged.lower():
            return 'dialogue'
        return 'final_response'

    def chat(self, messages, num_responses=1, temperature=None, top_p=None, max_tokens=None, debug=False):
        stage = self._infer_stage(messages)
        resolved_temperature = self.dialogue_temperature if stage == 'dialogue' else self.final_temperature
        resolved_max_tokens = self.dialogue_max_tokens if stage == 'dialogue' else self.final_max_tokens

        if temperature is not None:
            resolved_temperature = float(temperature)
        if max_tokens is not None:
            resolved_max_tokens = int(max_tokens)
        resolved_top_p = self.top_p if top_p is None else float(top_p)

        n = max(1, int(num_responses))
        call_idx = len(self.call_log)
        params_list = [
            SamplingParams(
                temperature=resolved_temperature,
                top_p=resolved_top_p,
                max_tokens=resolved_max_tokens,
                seed=self.base_seed + call_idx * 1000 + i,
            )
            for i in range(n)
        ]
        prompt_batch = [messages for _ in range(n)]

        try:
            results = self.model.chat(messages=prompt_batch, sampling_params=params_list, use_tqdm=False)
        except TypeError:
            results = self.model.chat(messages=prompt_batch, sampling_params=params_list)

        texts = []
        parsed = []
        for result in results:
            text = result.outputs[0].text if getattr(result, 'outputs', None) else str(result)
            text = text.strip()
            texts.append(text)
            parsed.append(maybe_parse_json_object(text))

        record = {
            'stage': stage,
            'messages': [dict(msg) for msg in messages],
            'temperature': resolved_temperature,
            'top_p': resolved_top_p,
            'max_tokens': resolved_max_tokens,
            'texts': texts,
            'parsed': parsed,
        }
        self.call_log.append(record)

        if debug:
            print(f'=== {self.name} / stage={stage} ===')
            show_messages(messages)
            for idx, text in enumerate(texts):
                print(f'--- raw output {idx} ---')
                print(text)
                print()

        return texts



def show_agent_calls(agent, last_n=None, show_prompt=False):
    calls = agent.call_log if last_n is None else agent.call_log[-last_n:]
    start_idx = 0 if last_n is None else max(0, len(agent.call_log) - len(calls))

    for offset, call in enumerate(calls, start=start_idx):
        print(f'===== {agent.name} / call {offset} / stage={call["stage"]} =====')
        print(f"temperature={call['temperature']} top_p={call['top_p']} max_tokens={call['max_tokens']}")
        if show_prompt:
            print()
            print('--- prompt ---')
            show_messages(call['messages'])
        for idx, text in enumerate(call['texts']):
            print(f'--- raw output {idx} ---')
            print(text)
            parsed = call['parsed'][idx]
            if parsed is not None:
                print()
                print('parsed json:')
                pprint(parsed)
            print()
        print()
    

In [ ]:
llm = LLM(
    model=MODEL_NAME,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    trust_remote_code=TRUST_REMOTE_CODE,
    seed=SEED,
)
        

In [ ]:
def make_env(
    seed=SEED,
    scenario_name=SCENARIO_NAME,
    private_profile_name=PRIVATE_PROFILE_NAME,
    generated_dialogue_turns=GENERATED_DIALOGUE_TURNS,
):
    candidate = VLLMInterviewAgent(
        name='Candidate',
        model=llm,
        model_name=MODEL_NAME,
        base_seed=seed,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        dialogue_temperature=DIALOGUE_TEMPERATURE,
        final_temperature=FINAL_TEMPERATURE,
        top_p=TOP_P,
        dialogue_max_tokens=DIALOGUE_MAX_TOKENS,
        final_max_tokens=FINAL_MAX_TOKENS,
    )
    interviewer = VLLMInterviewAgent(
        name='HiringManager',
        model=llm,
        model_name=MODEL_NAME,
        base_seed=seed + 10_000,
        reasoning_instruction='COD',
        instruction_format='reasoning',
        dialogue_temperature=DIALOGUE_TEMPERATURE,
        final_temperature=FINAL_TEMPERATURE,
        top_p=TOP_P,
        dialogue_max_tokens=DIALOGUE_MAX_TOKENS,
        final_max_tokens=FINAL_MAX_TOKENS,
    )
    spec = InterviewSpec(
        auto_generate_dialogue=True,
        generated_dialogue_turns=generated_dialogue_turns,
    )
    env = JobInterviewEnvironment(
        agents=[candidate, interviewer],
        seed=seed,
        scenario_name=scenario_name,
        private_profile_name=private_profile_name,
        spec=spec,
        history_window=HISTORY_WINDOW,
    )
    return env, candidate, interviewer
        

In [ ]:
env, candidate_agent, interviewer_agent = make_env()
state = env.get_state(include_system_prompt=True)

print('phase =', state['phase'])
print('scenario =', state['scenario']['name'])
print('private_profile =', state['hidden_state']['private_profile_name'])
print('candidate_target_total_comp =', state['hidden_state']['candidate_target_total_comp'])
print('dialogue_turns_in_history =', len(state['dialogue_history']))
        

In [ ]:
print('=== GENERATED PRE-OFFER DIALOGUE ===')
print()
show_dialogue(state['dialogue_history'][:-1])

print('=== FORMAL OFFER ===')
print()
show_dialogue(state['dialogue_history'][-1:])
    

In [ ]:
print('=== INTERVIEWER MODEL CALLS USED FOR DIALOGUE ===')
print()
show_agent_calls(interviewer_agent, show_prompt=False)

print('=== CANDIDATE MODEL CALLS USED FOR DIALOGUE ===')
print()
show_agent_calls(candidate_agent, show_prompt=False)
    

In [ ]:
print('=== FINAL CANDIDATE PROMPT ===')
print()
show_messages(state['messages'])
    

In [ ]:
result = env.step(num_responses=1, debug=False)

print('=== RAW FINAL CANDIDATE COMPLETION ===')
print()
print(candidate_agent.call_log[-1]['texts'][0])

print()
print('=== PARSED FINAL ACTION ===')
print()
pprint(env.last_candidate_action)

print()
print('=== LABEL ===')
print()
pprint(result['label'])

print()
print('=== RESOLUTION ===')
print()
pprint(result['resolution'])
    

In [ ]:
# Optional: rerun another hidden-truth case with the same loaded model.
# env2, candidate2, interviewer2 = make_env(seed=2, scenario_name='applied_research_scientist', private_profile_name='higher_offer')
# state2 = env2.get_state(include_system_prompt=True)
# show_dialogue(state2['dialogue_history'])
        